### NOTE 
- Previous version has both earliest_date, latest_day together with the binary flag, seems a little redundent (confirmed here in this notebook)
- To satisfy the fair AI review, all race columns have been dropped

In [ ]:
import sys
import pandas as pd

sys.path.append('../')
sys.path.append('../../src')

df = pd.read_parquet('../depression_all_features.parquet')
df.shape

In [ ]:
df_pivoted = df.pivot(index='birthid', columns='feature_name', values='value').reset_index()
df_pivoted.columns.name = ''
df_pivoted

In [ ]:
cols = [x for x in df_pivoted.columns if x.endswith('_earliest_day') or x.endswith('_latest_day')]
cols

In [ ]:
df_pivoted[cols].max().max()

In [ ]:
from preprocess import preprocess

preprocess(df_pivoted)

In [ ]:
df_pivoted.head()

In [ ]:
condition_1 = df_pivoted['edinburgh_max'].isna() == False
condition_2 = df_pivoted['phq9_total_max'].isna() == False
df_pivoted_screened = df_pivoted[ condition_1 | condition_2 ].copy()
df_pivoted_screened.shape

In [ ]:
cols_to_drop = ['mother_is_white', 'mother_is_black', 'mother_is_hispanic',
                'F53_label', 'edinburgh_max','phq9_total_max', 'birthid', 'target_variable', 'other_dx_code']

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from model_training import loop_over_selector
from feature_selectors import CachedTopKSelector

selector_basemodel = RandomForestClassifier(class_weight='balanced', random_state=42, min_samples_split=100)

pipeline = Pipeline([
    ('selector', CachedTopKSelector(selector_basemodel, method='shap')),
    ('clf', RandomForestClassifier(class_weight='balanced', random_state=42))
])

param_grid = {
    'selector__k': [10, 20, 30, 50, 100, 200, 300, 500, 1000, 3000, 5000],
    'clf__min_samples_split': [5, 20, 50, 70, 100, 200, 500, 700, 1000]
}

X = df_pivoted_screened.drop(columns=cols_to_drop)
y = df_pivoted_screened['target_variable']

metrics = loop_over_selector(X, y, pipeline, param_grid, outer_splits=5, inner_splits=5,
                             scoring='roc_auc', n_jobs=1)

In [ ]:
metrics

In [ ]:
from matplotlib import pyplot as plt

fig, axe = plt.subplots(1, 1, figsize=(8, 5))
tmp = metrics.groupby('selector__k')[['number_of_features', 'roc_auc']].agg(['mean', 'std'])
ks = tmp['number_of_features']['mean']
axe.errorbar(ks, tmp['roc_auc']['mean'], tmp['roc_auc']['std'], fmt='o-', capsize=5, linewidth=1.0)
axe.set_xlabel('Number of Features')
axe.set_ylabel('ROC_AUC')
axe.set_ylim([0.65, 0.75])
axe.set_xlim([0, 1000])

### 2 train without days

In [ ]:
cols_to_drop_including_days = cols_to_drop[:]
for col in [x for x in df_pivoted_screened.columns if x.endswith('_earliest_day') or x.endswith('_latest_day')]:
    cols_to_drop_including_days.append(col)
cols_to_drop_including_days

In [ ]:
X = df_pivoted_screened.drop(columns=cols_to_drop_including_days)
y = df_pivoted_screened['target_variable']

metrics_2 = loop_over_selector(X, y, pipeline, param_grid, outer_splits=5, inner_splits=5,
                               scoring='roc_auc', n_jobs=1)

In [ ]:
fig, axe = plt.subplots(1, 1, figsize=(8, 5))
tmp2 = metrics_2.groupby('selector__k')[['number_of_features', 'roc_auc']].agg(['mean', 'std'])
ks2 = tmp2['number_of_features']['mean']
axe.errorbar(ks, tmp['roc_auc']['mean'], tmp['roc_auc']['std'], fmt='o-', capsize=5, linewidth=1.0, label='with day')
axe.errorbar(ks2, tmp2['roc_auc']['mean'], tmp2['roc_auc']['std'], fmt='kx-', capsize=5, linewidth=1.0, label='without day')
axe.set_xlabel('Number of Features')
axe.set_ylabel('ROC_AUC')
axe.set_ylim([0.65, 0.75])
axe.set_xlim([0, 1000])
axe.legend()

In [ ]:
fig, axe = plt.subplots(1, 1, figsize=(8, 5))
tmp2 = metrics_2.groupby('selector__k')[['number_of_features', 'roc_auc']].agg(['mean', 'std'])
ks2 = tmp2['number_of_features']['mean']
axe.errorbar(ks, tmp['roc_auc']['mean'], tmp['roc_auc']['std'], fmt='o-', capsize=5, linewidth=1.0, label='with day')
axe.errorbar(ks2, tmp2['roc_auc']['mean'], tmp2['roc_auc']['std'], fmt='kx-', capsize=5, linewidth=1.0, label='without day')
axe.set_xlabel('Number of Features')
axe.set_ylabel('ROC_AUC')
axe.set_ylim([0.65, 0.75])
axe.set_xlim([0, 200])
axe.legend()

### as expected, better to just train it without the day, especially when low number of features are allowed.

### 3. output the final model with 20 features

In [ ]:
metrics_2.loc[metrics_2['selector__k']==20, 'selected_features'].tolist()

In [ ]:
from collections import defaultdict

counts = defaultdict(lambda:0)
for features in metrics_2.loc[metrics_2['selector__k']==20, 'selected_features'].tolist():
    for feature in features:
        counts[feature] += 1
sorted_counts = sorted(counts.items(), key=lambda x: x[1], reverse=True)
sorted_counts

In [ ]:
sorted_counts[:21]

In [ ]:
selected_features = [
    'phq_or_edinburgh_max_21012953',
    'mother_age_at_birth',
    'phq_or_edinburgh_max_21012959',
    'O99_342_has_dx',
    'phq_or_edinburgh_max_21012951',
    'F41_9_has_dx',
    'phq_or_edinburgh_max_21012949',
    'O99_343_has_dx',
    'F32_9_has_dx',
    'O99_344_has_dx',
    'phq_or_edinburgh_max_99046-5',
    'total_visits_during_pregnancy_count',
    'phq_or_edinburgh_max_21012950',
    'R10_9_has_dx',
    'Z86_59_has_dx',
    'mother_height',
    'enc_type_OT_during_pregnancy_count',
    'smoking',
    'phq_or_edinburgh_max_21012948',
    'F32_A_has_dx'
]

In [ ]:
from model_training import nested_cv_evaluation_feature_selection

X = df_pivoted_screened[selected_features]
print(X.shape)
y = df_pivoted_screened['target_variable']
clf = RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced')
param_grid = {'min_samples_split': [20, 30, 50, 70, 100, 150, 200, 250, 300, 500]}

metrics_3 = nested_cv_evaluation_feature_selection(X, y, clf, param_grid,
                                                   outer_splits=5, inner_splits=5, random_state=42)

In [ ]:
metrics_3

In [ ]:
import json

metadata = {
    "min_sample_splits": 150,
    "n_estimators": 200,
    "roc_auc_avg": metrics_3['roc_auc'].mean(),
    "roc_auc_std": metrics_3['roc_auc'].std(),
    "selected_features": selected_features
}

with open("../models/model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)


In [ ]:
clf = RandomForestClassifier(n_estimators=metadata['n_estimators'],
                             min_samples_split=metadata['min_sample_splits'],
                             random_state=42,
                             class_weight='balanced')
clf.fit(X, y)

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y, clf.predict_proba(X)[:,1])

In [ ]:
import pickle

with open("../models/ppd_20_features.pkl", "wb") as fid:
    pickle.dump(clf, fid)


In [ ]:
with open("../models/ppd_20_features.pkl", "rb") as fid:
    clf_loaded = pickle.load(fid)

roc_auc_score(y, clf_loaded.predict_proba(X)[:,1])